# Lookahead bias: the backtest bug you can't see

A fundamental backtest has a quiet failure mode: it uses numbers **before they were public**.
Annual revenue for a fiscal year ending September 30 doesn't exist on September 30 — it exists
when the 10-K is filed, weeks later. Join your signal on the period-end date and every
fundamental strategy you test gets a small, systematic peek at the future.

This notebook shows the bug on real data, then measures it. It runs end-to-end with
**no API key** — it uses the public Tradevo Data proof pack (5 US companies, 16 annual concepts,
the latest 3 fiscal years, every value stamped with the date it first became public).

Honest scope, up front: the proof pack is **annual-only** (10-K/10-K/A), **US-only**,
and intentionally tiny. The hosted API adds quarterly 10-Q history while remaining US-only.

In [ ]:
# On Colab (or any fresh environment) this installs the client; otherwise it's a no-op.
try:
    import tradevodata
except ImportError:
    %pip install -q tradevodata pandas
    import tradevodata

import pandas as pd

print("tradevodata", tradevodata.__version__)

## Load the public proof pack

`tv.sample()` downloads the public CSV once and caches it in your temp directory. No key,
no signup.

In [ ]:
import tradevodata as tv

df = tv.sample()
print(f"{len(df):,} rows · {df['ticker'].nunique()} companies · {df['concept'].nunique()} concepts")
df.head()

Two columns carry the whole point-in-time promise:

- **`period_end`** — when the fiscal period ended (what a period-keyed dataset indexes by),
- **`first_filed`** — when the numbers actually became public (the 10-K filing date).

`lag_days` is the gap between them. Everything a backtest is allowed to know on a date is
determined by `first_filed`, not `period_end`.

## The experiment: naive join vs honest join

For a few tickers, ask the same question at every month-end: *what is the latest annual
revenue I know right now?* Two ways to answer it:

- **Naive** — join on `period_end`. This is what you get by default from a dataset keyed by period end, and it's wrong: it assumes the numbers appear the moment the fiscal year closes.
- **Honest** — join on `first_filed`. Numbers only become visible once the 10-K is filed.

Same data, same code — only the join key differs.

In [ ]:
TICKERS = ["AAPL", "NVDA", "WMT"]

rev = df[(df["concept"] == "Revenue") & (df["ticker"].isin(TICKERS))].copy()

# Every month-end across the sample's history (period_range keeps this
# compatible with both older and newer pandas).
month_ends = (
    pd.period_range(rev["period_end"].min(), rev["first_filed"].max(), freq="M")
    .to_timestamp(how="end")
    .normalize()
)

def latest_known(join_col):
    """Latest annual revenue per ticker at each month-end, joining on join_col."""
    frames = []
    for t in TICKERS:
        r = (
            rev[rev["ticker"] == t]
            .sort_values(join_col)[[join_col, "fiscal_year", "original_value"]]
            .rename(columns={join_col: "known_from", "original_value": "revenue"})
        )
        grid = pd.DataFrame({"month_end": month_ends})
        m = pd.merge_asof(grid, r, left_on="month_end", right_on="known_from")
        m.insert(0, "ticker", t)
        frames.append(m)
    return pd.concat(frames, ignore_index=True)

naive = latest_known("period_end")    # the bug
honest = latest_known("first_filed")  # reality

## Where the naive join "knows" the future

Every ticker-month where the two joins disagree is a month where the naive backtest is
trading on a number that **was not public yet**.

In [ ]:
cmp = naive[["ticker", "month_end"]].copy()
cmp["naive_fy"] = naive["fiscal_year"]
cmp["honest_fy"] = honest["fiscal_year"]
cmp["naive_revenue"] = naive["revenue"]
cmp["honest_revenue"] = honest["revenue"]

phantom = cmp[cmp["naive_fy"].notna() & (cmp["naive_fy"] != cmp["honest_fy"])]

n_live = len(cmp[cmp["naive_fy"].notna()])
print(
    f"{len(phantom)} of {n_live} ticker-months ({len(phantom) / n_live:.0%}) "
    "use a number that was not yet public."
)
phantom.tail(6)

Concrete case: Apple's fiscal 2024 ended **2024-09-28**, but the 10-K wasn't filed until
**2024-11-01**. At the September and October month-ends the naive join already "knows"
FY2024 revenue ($391.0B) while the honest join still shows FY2023 ($383.3B) — because that
was, truthfully, all anyone knew:

In [ ]:
aapl_2024 = phantom[
    (phantom["ticker"] == "AAPL") & (phantom["month_end"].dt.year == 2024)
]
aapl_2024

Is a ~$8B revenue peek harmless? No — the phantom window is exactly when the *new* number
moves prices (earnings season). A value or quality signal rebalanced monthly on the naive
join gets the fresh fundamentals 1–2 rebalances early, every year, for every stock. That's
not noise; it's a systematic, always-favorable head start that inflates the backtest and
vanishes in live trading.

## How big is the head start?

`lag_days` measures it directly, per row. We quantify on the rows with reliable filing
dates (the proof pack flags the one row where the filing date is outside the normal range —
we measure honestly or not at all):

In [ ]:
reliable = df[df["filed_reliable"]]

print(f"Reliable-filing-date rows: {len(reliable):,} of {len(df):,}")
print(
    f"Fundamentals became public on average {reliable['lag_days'].mean():.0f} days "
    f"after the period ended (median {reliable['lag_days'].median():.0f}, "
    f"max {reliable['lag_days'].max():.0f})."
)
reliable["lag_days"].describe().round(1)

**Mean 35.2 days early, max 48 in this proof pack.** These five companies have
disciplined filing calendars, so this is not a proxy for the whole universe. Across the full 5,000+ company universe the measured mean lag is
**66 days** (median 60, 90th percentile 90 (rows are QA-capped at 120, so 120 is a cutoff, not a measurement)): smaller companies file later, so the naive join's future-peek
is *worse* exactly where backtests are most fragile.

One more honesty dimension, since the proof pack carries it: `original_value` is what was
first reported, `latest_value` is the current revision, and `restated` flags material
changes. A backtest should see the number as it was known then — not as it was quietly
rewritten later:

In [ ]:
restated = df[df["restated"]]
print(f"{len(restated)} of {len(df):,} rows were later restated by more than 0.5%. Example:")
restated[["ticker", "concept", "fiscal_year", "original_value", "latest_value"]].head(3)

## The fix

Join on `first_filed`. That's it — the `honest` frame above is a correct point-in-time
view, and on this public proof pack you can build one for five companies right now.

## The full universe

The paid API serves the same semantics for **5,000+ US companies / 600,000+ annual point-in-time
rows** (live totals: [tradevodata.com/status](https://tradevodata.com/status)), with the `as_of` logic done server-side — ask what was knowable about a ticker on a
date, get exactly that:

In [ ]:
# from tradevodata import Client
#
# client = Client(api_key="tvd_...")   # or: export TRADEVODATA_API_KEY=tvd_...
# client.fundamentals("AAPL", as_of="2024-06-30")
#
# # As of June 2024 that returns FY2023 numbers — FY2024 wasn't filed until
# # November. Nothing you couldn't have known. That's the product.

**[tradevodata.com](https://tradevodata.com/?utm_source=colab&utm_medium=notebook&utm_campaign=pit-proof-2026-08)** — start a card-backed 30-day evaluation for up to 50 companies; cancel before day 30 and pay $0, otherwise Pro renews at $29/month. Docs: [tradevodata.com/docs](https://tradevodata.com/docs?utm_source=colab&utm_medium=notebook&utm_campaign=pit-proof-2026-08) · public proof pack +
methodology: [github.com/christianpichichero-max/pit-fundamentals](https://github.com/christianpichichero-max/pit-fundamentals).

Same honest limits as everywhere else: US-only and no delisted coverage. The proof pack is annual-only; the hosted API adds quarterly history. Evaluation access is 500 requests/day across 50 companies with no bulk; Pro adds 5,000 requests/day and bulk.